# Image Matching Pipeline — Pairwise Embedding Comparison + Classification
---
Pipeline นี้เทียบ image embeddings (CLIP + ArcFace) แบบ **pairwise ก่อน** แล้วค่อยเข้า classification

| Step | ชื่อ | หน้าที่ |
|------|------|----------|
| 1 | Configuration & Imports | ตั้งค่า paths, import libraries |
| 2 | Data Structures | สร้าง ImageProfile dataclass |
| 3 | Embedding Loader | Scan directories, โหลด .npy files |
| 4 | Similarity Functions | คำนวณ cosine similarity, L2 distance |
| 5 | **Pairwise Comparison** | **เทียบ embeddings ทั้งหมดแบบ all-vs-all** |
| 6 | Candidate Selection | เลือก Top-K ที่คล้ายที่สุด |
| 7 | Feature Building | สร้าง feature vectors สำหรับ classification |
| 8 | Training | Train Logistic Regression + Random Forest |
| 9 | Evaluation | Top-1 accuracy + Candidate recall |
| 10 | Predict Similarity | เทียบ 2 profiles เฉพาะ |

---
## Step 1: Configuration & Imports
- Import libraries ที่จำเป็น
- กำหนด paths ไปยัง embedding directories
- ตั้งค่า dimensions (CLIP=512, ArcFace=512, Fused=1024)

In [2]:
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ============================================================
# CONFIGURATION
# ============================================================

SCRIPT_DIR = Path('.').absolute()  # directory ที่ notebook อยู่
DATA_DIR = SCRIPT_DIR.parent / 'data'

DIRS = {
    'embeddings_clip': DATA_DIR / 'embeddings' / 'clip',
    'embeddings_face': DATA_DIR / 'embeddings' / 'face',
    'results': DATA_DIR / 'processed' / 'image_pipeline_results',
}

CLIP_DIM = 512
ARCFACE_DIM = 512
FUSED_DIM = CLIP_DIM + ARCFACE_DIM  # 1024

TOP_K = 20           # Top-K matches ต่อ profile
CANDIDATE_PCT = 0.10 # Candidate pool = 10%
RANDOM_STATE = 42

print('Configuration loaded')
print(f'  CLIP dir:    {DIRS["embeddings_clip"]}')
print(f'  ArcFace dir: {DIRS["embeddings_face"]}')
print(f'  Results dir: {DIRS["results"]}')
print(f'  Fused dim:   {FUSED_DIM}')

Configuration loaded
  CLIP dir:    d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\embeddings\clip
  ArcFace dir: d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\embeddings\face
  Results dir: d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\processed\image_pipeline_results
  Fused dim:   1024


---
## Step 2: Data Structures
- `ImageProfile` — เก็บข้อมูล profile + embeddings
- `log_stage()` — แสดงเวลาทำงาน

In [3]:
@dataclass
class ImageProfile:
    profile_id: str
    platform: str
    clip_emb: Optional[np.ndarray] = None      # CLIP embedding (512-dim)
    face_emb: Optional[np.ndarray] = None       # ArcFace embedding (512-dim)
    fused_emb: Optional[np.ndarray] = None      # Fused [clip; face] (1024-dim)
    has_clip: bool = False
    has_face: bool = False


def log_stage(stage: str, start_ts: float) -> None:
    elapsed = time.time() - start_ts
    print(f'  [{stage}] done in {elapsed:.2f}s')

print('Data structures defined')

Data structures defined


---
## Step 3: Embedding Loader — Scan directories & โหลด embeddings
- `load_and_normalize()` — โหลด .npy file แล้ว normalize เป็น unit vector
- `fuse_embeddings()` — รวม CLIP + ArcFace → 1024-dim vector
- `discover_profiles()` — Scan directories เพื่อค้นหา profiles ทั้งหมด

**ไม่ต้องพึ่ง pair CSV อีกแล้ว** — แค่ scan จาก embedding directories เลย

In [4]:
def load_and_normalize(npy_path: Path) -> Optional[np.ndarray]:
    if not npy_path.exists():
        return None
    emb = np.load(npy_path).flatten().astype(np.float32)
    norm = np.linalg.norm(emb)
    if norm > 0:
        emb = emb / norm
    return emb


def fuse_embeddings(clip_emb, face_emb):
    if clip_emb is None and face_emb is None:
        return None
    if clip_emb is None:
        clip_emb = np.zeros(CLIP_DIM, dtype=np.float32)
    if face_emb is None:
        face_emb = np.zeros(ARCFACE_DIM, dtype=np.float32)
    fused = np.concatenate([clip_emb, face_emb])
    norm = np.linalg.norm(fused)
    if norm > 0:
        fused = fused / norm
    return fused


def discover_profiles() -> List[ImageProfile]:
    clip_dir = DIRS['embeddings_clip']
    face_dir = DIRS['embeddings_face']
    seen = {}
    for emb_dir, emb_type in [(clip_dir, 'clip'), (face_dir, 'face')]:
        if not emb_dir.exists():
            print(f'  Warning: {emb_dir} not found')
            continue
        for npy_file in emb_dir.glob('*.npy'):
            stem = npy_file.stem
            parts = stem.rsplit('_', 1)
            if len(parts) != 2:
                continue
            pid, plat = parts
            key = (pid, plat)
            if key not in seen:
                seen[key] = {}
            seen[key][emb_type] = npy_file

    profiles = []
    for (pid, plat), paths in seen.items():
        clip_emb = load_and_normalize(paths['clip']) if 'clip' in paths else None
        face_emb = load_and_normalize(paths['face']) if 'face' in paths else None
        fused_emb = fuse_embeddings(clip_emb, face_emb)
        if clip_emb is None and face_emb is None:
            continue
        profiles.append(ImageProfile(
            profile_id=pid, platform=plat,
            clip_emb=clip_emb, face_emb=face_emb, fused_emb=fused_emb,
            has_clip=clip_emb is not None, has_face=face_emb is not None,
        ))
    profiles.sort(key=lambda p: (p.profile_id, p.platform))
    return profiles

print('Embedding loader functions defined')

Embedding loader functions defined


### ▶ รัน: โหลด Embeddings ทั้งหมด

In [5]:
step_ts = time.time()
profiles = discover_profiles()
n_profiles = len(profiles)

clip_count = sum(1 for p in profiles if p.has_clip)
face_count = sum(1 for p in profiles if p.has_face)
both_count = sum(1 for p in profiles if p.has_clip and p.has_face)
platforms = set(p.platform for p in profiles)

print(f'Profiles loaded: {n_profiles}')
print(f'  With CLIP:    {clip_count}')
print(f'  With ArcFace: {face_count}')
print(f'  With both:    {both_count}')
print(f'  Platforms:    {platforms}')
log_stage('load_embeddings', step_ts)

Profiles loaded: 2459
  With CLIP:    2459
  With ArcFace: 2459
  With both:    2459
  Platforms:    {'twitter'}
  [load_embeddings] done in 95.07s


---
## Step 4: Similarity Functions
- `cosine_similarity()` — Cosine similarity (dot product เพราะ normalized)
- `l2_distance()` — Euclidean distance

In [6]:
def cosine_similarity(a, b):
    if a is None or b is None:
        return 0.0
    return float(max(-1.0, min(1.0, np.dot(a, b))))


def l2_distance(a, b):
    if a is None or b is None:
        return 2.0
    return float(np.linalg.norm(a - b))

print('Similarity functions defined')

Similarity functions defined


---
## Step 5: Pairwise Comparison ⭐ (ขั้นตอนหลัก)
**เทียบ embeddings ทั้งหมดแบบ all-vs-all** — ทำ **ก่อน** classification

วิธี:
1. สร้าง fused matrix (N x 1024)
2. คำนวณ cosine similarity ด้วย matrix multiplication: `matrix @ matrix.T`
3. ได้ similarity matrix (N x N)

In [7]:
def compute_pairwise_similarity(profiles):
    n = len(profiles)
    fused_matrix = np.zeros((n, FUSED_DIM), dtype=np.float32)
    for i, p in enumerate(profiles):
        if p.fused_emb is not None:
            fused_matrix[i] = p.fused_emb
    sim_matrix = fused_matrix @ fused_matrix.T
    np.clip(sim_matrix, -1.0, 1.0, out=sim_matrix)
    return sim_matrix

print('Pairwise comparison function defined')

Pairwise comparison function defined


### ▶ รัน: เทียบ Pairwise ทั้งหมด

In [8]:
step_ts = time.time()
sim_matrix = compute_pairwise_similarity(profiles)

upper_tri = sim_matrix[np.triu_indices(n_profiles, k=1)]
print(f'Pairwise similarity computed: {n_profiles} x {n_profiles} = {n_profiles*n_profiles:,} pairs')
print(f'  Mean:   {upper_tri.mean():.4f}')
print(f'  Std:    {upper_tri.std():.4f}')
print(f'  Min:    {upper_tri.min():.4f}')
print(f'  Max:    {upper_tri.max():.4f}')
print(f'  Median: {np.median(upper_tri):.4f}')
log_stage('pairwise_similarity', step_ts)

Pairwise similarity computed: 2459 x 2459 = 6,046,681 pairs
  Mean:   0.4527
  Std:    0.2017
  Min:    0.0051
  Max:    1.0000
  Median: 0.3710
  [pairwise_similarity] done in 0.20s


### ▶ รัน: สร้าง Top-K matches แล้วบันทึก
สำหรับแต่ละ profile → เก็บ Top-K คนที่คล้ายที่สุด พร้อม scores

In [9]:
def build_pairwise_results(profiles, sim_matrix, top_k):
    rows = []
    n = len(profiles)
    for i in range(n):
        src = profiles[i]
        sims = sim_matrix[i].copy()
        sims[i] = -2.0
        k = min(top_k, n - 1)
        top_indices = np.argpartition(sims, -k)[-k:]
        top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]
        for rank, j in enumerate(top_indices, start=1):
            tgt = profiles[j]
            rows.append({
                'source_id': src.profile_id,
                'source_platform': src.platform,
                'rank': rank,
                'target_id': tgt.profile_id,
                'target_platform': tgt.platform,
                'fused_cosine': round(float(sims[j]), 6),
                'clip_cosine': round(cosine_similarity(src.clip_emb, tgt.clip_emb), 6),
                'face_cosine': round(cosine_similarity(src.face_emb, tgt.face_emb), 6),
                'clip_l2': round(l2_distance(src.clip_emb, tgt.clip_emb), 6),
                'face_l2': round(l2_distance(src.face_emb, tgt.face_emb), 6),
                'is_same_person': src.profile_id == tgt.profile_id,
            })
    return pd.DataFrame(rows)


step_ts = time.time()
DIRS['results'].mkdir(parents=True, exist_ok=True)

pairwise_df = build_pairwise_results(profiles, sim_matrix, TOP_K)
pairwise_out = DIRS['results'] / 'pairwise_top_matches.csv'
pairwise_df.to_csv(pairwise_out, index=False)

print(f'Saved: {pairwise_out}')
print(f'  Total rows: {len(pairwise_df):,}')
log_stage('save_pairwise', step_ts)

print('\nSample Top-1 matches:')
pairwise_df[pairwise_df['rank'] == 1].head(10)

Saved: d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\processed\image_pipeline_results\pairwise_top_matches.csv
  Total rows: 49,180
  [save_pairwise] done in 1.32s

Sample Top-1 matches:


,source_id,source_platform,rank,target_id,target_platform,fused_cosine,clip_cosine,face_cosine,clip_l2,face_l2,is_same_person
0,aajain,twitter,1,nataliamartinez,twitter,0.562816,0.780475,0.345157,0.662609,1.144415,False
20,aaronbavs,twitter,1,johndeguzman,twitter,0.876057,0.763144,0.988970,0.688267,0.148528,False
40,abahgat,twitter,1,jiteshpanchal,twitter,0.838076,0.880954,0.795199,0.487947,0.640002,False
60,abdisk,twitter,1,avinashconda,twitter,0.572573,0.811554,0.333592,0.613916,1.154477,False
80,abdullawissam,twitter,1,ranjiththomas,twitter,0.602171,0.832696,0.371646,0.578454,1.121030,False
100,abo1bs,twitter,1,vicaso,twitter,0.926586,0.861495,0.991677,0.526317,0.129022,False
120,acanadianfoodie,twitter,1,gwynnekostin,twitter,0.899826,0.841378,0.958274,0.563245,0.288881,False
140,acarrillo,twitter,1,jeansanchez,twitter,0.894061,0.803654,0.984467,0.626651,0.176255,False
160,ackleysuicide,twitter,1,anamagal,twitter,0.879951,0.840860,0.919041,0.564164,0.402390,False
180,actifirmskincare,twitter,1,beautyluxury,twitter,0.777050,0.621656,0.932443,0.869879,0.367578,False


---
## Step 6-7: Candidate Selection + Feature Building
เลือก Top-K candidates จาก similarity matrix แล้วสร้าง 7 features:

| # | Feature | ความหมาย |
|---|---------|----------|
| 0 | clip_cosine | Cosine similarity ของ CLIP |
| 1 | face_cosine | Cosine similarity ของ ArcFace |
| 2 | fused_cosine | Cosine similarity ของ Fused |
| 3 | clip_l2 | L2 distance ของ CLIP |
| 4 | face_l2 | L2 distance ของ ArcFace |
| 5 | has_both_clip | ทั้งคู่มี CLIP (1/0) |
| 6 | has_both_face | ทั้งคู่มี ArcFace (1/0) |

In [10]:
def select_candidates_from_matrix(sim_matrix, src_idx, k):
    sims = sim_matrix[src_idx].copy()
    sims[src_idx] = -2.0
    k = min(k, len(sims) - 1)
    top_indices = np.argpartition(sims, -k)[-k:]
    top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]
    return top_indices.astype(np.int32), sims[top_indices]


def build_features_for_candidates(src, candidate_indices, all_profiles):
    n = len(candidate_indices)
    X = np.zeros((n, 7), dtype=np.float64)
    y = np.zeros(n, dtype=np.int8)
    for i, tgt_idx in enumerate(candidate_indices):
        tgt = all_profiles[int(tgt_idx)]
        X[i] = [
            cosine_similarity(src.clip_emb, tgt.clip_emb),
            cosine_similarity(src.face_emb, tgt.face_emb),
            cosine_similarity(src.fused_emb, tgt.fused_emb),
            l2_distance(src.clip_emb, tgt.clip_emb),
            l2_distance(src.face_emb, tgt.face_emb),
            1.0 if (src.has_clip and tgt.has_clip) else 0.0,
            1.0 if (src.has_face and tgt.has_face) else 0.0,
        ]
        y[i] = 1 if src.profile_id == tgt.profile_id else 0
    return X, y

print('Candidate selection & feature building defined')

Candidate selection & feature building defined


---
## Step 8: Training
- Balanced training: 1 positive + 1 hard negative ต่อ source
- Train **Logistic Regression** + **Random Forest**

In [11]:
def build_balanced_train_set(profiles, sim_matrix, train_indices, k, rng):
    X_list, y_list = [], []
    for src_i in train_indices:
        src = profiles[src_i]
        cand_indices, _ = select_candidates_from_matrix(sim_matrix, src_i, k)
        X_cand, y_cand = build_features_for_candidates(src, cand_indices, profiles)
        pos_mask = y_cand == 1
        neg_mask = y_cand == 0
        if not pos_mask.any() or not neg_mask.any():
            continue
        pos_idx = np.where(pos_mask)[0]
        best_pos = int(pos_idx[np.argmax(X_cand[pos_idx, 2])])
        neg_idx = np.where(neg_mask)[0]
        rand_neg = int(rng.choice(neg_idx))
        X_list.append(X_cand[[best_pos, rand_neg]])
        y_list.append(np.array([1, 0], dtype=np.int8))
    if not X_list:
        raise RuntimeError('No training samples')
    return np.vstack(X_list).astype(np.float64), np.concatenate(y_list).astype(np.int8)

print('Training function defined')

Training function defined


### ▶ รัน: Train/Test Split + Train Models

In [13]:
# --- Train/Test Split (60/40) ---
k_candidates = max(1, int(CANDIDATE_PCT * n_profiles))
all_indices = np.arange(n_profiles)
train_idx, test_idx = train_test_split(
    all_indices, test_size=0.4, random_state=RANDOM_STATE, shuffle=True,
)
print(f'Train: {len(train_idx)}, Test: {len(test_idx)}')
print(f'Candidate pool: top {CANDIDATE_PCT*100:.0f}% = {k_candidates} per profile')

# --- Build balanced training set ---
step_ts = time.time()
rng = np.random.default_rng(RANDOM_STATE)
X_train, y_train = build_balanced_train_set(
    profiles, sim_matrix, train_idx, k_candidates, rng,
)
print(f'\nTraining samples: {len(X_train)} (pos={y_train.sum()}, neg={len(y_train)-y_train.sum()})')
log_stage('build_trainset', step_ts)

# --- Train Logistic Regression ---
lr_ts = time.time()
lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
)
lr_model.fit(X_train, y_train)
log_stage('train_logistic_regression', lr_ts)

# --- Train Random Forest ---
rf_ts = time.time()
rf_model = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced',
)
rf_model.fit(X_train, y_train)
log_stage('train_random_forest', rf_ts)

print('\nBoth models trained!')

Train: 1475, Test: 984
Candidate pool: top 10% = 245 per profile


RuntimeError: No training samples

---
## Step 9: Evaluation
- **Top-1 Accuracy**: model ทำนาย profile ที่คล้ายที่สุดถูกต้องกี่ %
- **Candidate Recall**: true match อยู่ใน top-K candidates กี่ %

In [ ]:
def evaluate_top1(model, model_name, profiles, sim_matrix, test_indices, k):
    rows = []
    correct, total, cand_recall = 0, 0, 0
    for src_i in test_indices:
        src = profiles[src_i]
        cand_indices, _ = select_candidates_from_matrix(sim_matrix, src_i, k)
        X_cand, y_cand = build_features_for_candidates(src, cand_indices, profiles)
        has_true = int((y_cand == 1).any())
        cand_recall += has_true
        if len(X_cand) == 0:
            continue
        probs = model.predict_proba(X_cand)[:, 1]
        probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        best_local = int(np.argmax(probs))
        best_tgt = profiles[int(cand_indices[best_local])]
        is_correct = 1 if best_tgt.profile_id == src.profile_id else 0
        correct += is_correct
        total += 1
        rows.append({
            'model': model_name, 'source_id': src.profile_id,
            'predicted_id': best_tgt.profile_id,
            'is_correct_top1': is_correct,
            'match_probability': round(float(probs[best_local]), 6),
            'fused_cosine': round(float(X_cand[best_local, 2]), 6),
            'true_in_topk': has_true,
        })
    acc = (correct / total) if total else 0.0
    cr = (cand_recall / len(test_indices)) if len(test_indices) else 0.0
    metrics = {'model': model_name, 'test_size': len(test_indices),
               'top1_accuracy': round(acc, 6), 'candidate_recall_topk': round(cr, 6)}
    return pd.DataFrame(rows), metrics

print('Evaluation function defined')

### ▶ รัน: Evaluate ทั้ง 2 models

In [ ]:
step_ts = time.time()

lr_df, lr_metrics = evaluate_top1(
    lr_model, 'LogisticRegression', profiles, sim_matrix, test_idx, k_candidates,
)
rf_df, rf_metrics = evaluate_top1(
    rf_model, 'RandomForest', profiles, sim_matrix, test_idx, k_candidates,
)
log_stage('evaluation', step_ts)

print()
print('=' * 50)
print('RESULTS')
print('=' * 50)
print(f'  LogisticRegression:')
print(f'    Top-1 Accuracy:   {lr_metrics["top1_accuracy"]:.4f}')
print(f'    Candidate Recall: {lr_metrics["candidate_recall_topk"]:.4f}')
print(f'  RandomForest:')
print(f'    Top-1 Accuracy:   {rf_metrics["top1_accuracy"]:.4f}')
print(f'    Candidate Recall: {rf_metrics["candidate_recall_topk"]:.4f}')
print('=' * 50)

### ▶ รัน: บันทึกผลลัพธ์ทั้งหมด

In [ ]:
import joblib

lr_out = DIRS['results'] / 'predictions_logistic_regression.csv'
rf_out = DIRS['results'] / 'predictions_random_forest.csv'
lr_df.to_csv(lr_out, index=False)
rf_df.to_csv(rf_out, index=False)

metrics_df = pd.DataFrame([lr_metrics, rf_metrics])
metrics_out = DIRS['results'] / 'metrics_summary.csv'
metrics_df.to_csv(metrics_out, index=False)

model_out = DIRS['results'] / 'rf_model.joblib'
joblib.dump(rf_model, model_out)

print('All results saved:')
print(f'  {pairwise_out}')
print(f'  {lr_out}')
print(f'  {rf_out}')
print(f'  {metrics_out}')
print(f'  {model_out}')

---
## Step 10: Predict Similarity
ใส่ชื่อ 2 profiles แล้วดูว่าเป็นคนเดียวกันไหม

In [1]:
def predict_similarity(profile_a_id, profile_b_id, platform_a='twitter', platform_b='twitter', model=None):
    clip_a = load_and_normalize(DIRS['embeddings_clip'] / f'{profile_a_id}_{platform_a}.npy')
    clip_b = load_and_normalize(DIRS['embeddings_clip'] / f'{profile_b_id}_{platform_b}.npy')
    face_a = load_and_normalize(DIRS['embeddings_face'] / f'{profile_a_id}_{platform_a}.npy')
    face_b = load_and_normalize(DIRS['embeddings_face'] / f'{profile_b_id}_{platform_b}.npy')
    fused_a = fuse_embeddings(clip_a, face_a)
    fused_b = fuse_embeddings(clip_b, face_b)

    result = {
        'profile_a': f'{profile_a_id} ({platform_a})',
        'profile_b': f'{profile_b_id} ({platform_b})',
        'clip_cosine': round(cosine_similarity(clip_a, clip_b), 6),
        'face_cosine': round(cosine_similarity(face_a, face_b), 6),
        'fused_cosine': round(cosine_similarity(fused_a, fused_b), 6),
        'clip_l2': round(l2_distance(clip_a, clip_b), 6),
        'face_l2': round(l2_distance(face_a, face_b), 6),
    }
    if model is not None:
        features = np.array([[
            result['clip_cosine'], result['face_cosine'], result['fused_cosine'],
            result['clip_l2'], result['face_l2'],
            1.0 if (clip_a is not None and clip_b is not None) else 0.0,
            1.0 if (face_a is not None and face_b is not None) else 0.0,
        ]])
        prob = model.predict_proba(features)[0, 1]
        result['match_probability'] = round(float(prob), 6)
        result['is_same_person'] = bool(prob > 0.5)
    else:
        result['is_same_person'] = result['fused_cosine'] > 0.7
    return result


# ลองเทียบ 2 profiles (แก้ชื่อได้)
result = predict_similarity('adamthede', 'bioanarchism', model=rf_model)
print('Prediction result:')
for k, v in result.items():
    print(f'  {k}: {v}')

NameError: name 'rf_model' is not defined